# load file

In [1]:
from scipy.io import loadmat
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 

print(os.getcwd())

/Users/ernstdavidts/Library/CloudStorage/OneDrive-UGent/Master I/Oslo/Curved_Running_Pilot/Base/tests


In [2]:
data = loadmat("../files/ED_Shoe25OpenSim_python.mat")
print(data.keys())

angles = data['ANGLES_TABLE'][0,0]
print(angles.dtype.names)

angles_curve_1 = angles["x25_Curve_1_mat"]
print(type(angles_curve_1))

print(angles_curve_1.shape)


FileNotFoundError: [Errno 2] No such file or directory: '../files/ED_Shoe25OpenSim_python.mat'

In [ ]:
data_labels = loadmat("../files/angles_with_labels.mat")

raw_labels = data_labels["angle_labels"]

labels = [str(x[0]) for x in raw_labels[0]]

print(labels)

In [ ]:
angle_dict = {}

for n in range(len(labels)):
    angle_dict[labels[n]] = angles_curve_1[:,n]

In [ ]:
def get_angle_data(joint, trial="Curve_1", specification="Rotation", header="ANGLES_TABLE"):
    type_data = data[header][0,0]
    trial_data = type_data[f"x25_{trial}_mat"]

    angle_dict_local = {labels[n]: trial_data[:, n] for n in range(len(labels))}
    x = trial_data[:, 0]

    if joint in ["ankle", "knee"]:
        y_l = [angle_dict_local[k] for k in angle_dict_local.keys() if joint in k and k.endswith("_l")][0]
        y_r = [angle_dict_local[k] for k in angle_dict_local.keys() if joint in k and k.endswith("_r")][0]
        return x, y_l, y_r

    elif joint == "hip":
        y_l = [angle_dict_local[k] for k in angle_dict_local.keys() if specification.lower() in k and k.endswith("_l")][0]
        y_r = [angle_dict_local[k] for k in angle_dict_local.keys() if specification.lower() in k and k.endswith("_r")][0]
        return x, y_l, y_r

    else:
        y = [angle_dict_local[k] for k in angle_dict_local.keys() if joint in k and specification.lower() in k][0]
        return x, y

In [ ]:
def plot_angle(joint, trial="Curve_1", specification="Rotation"):
    result = get_angle_data(joint, trial, specification)

    if len(result) == 3:
        _, y_l, y_r = result
        if joint != "hip":
            label = "Angle"
        else: 
            label = f"{specification}"
        plt.plot(_, y_l, color="red", label="Left")
        plt.plot(_, y_r, color="blue", label="Right")
        plt.xlabel("Time (s)")
        plt.ylabel(label)
        plt.title(f"{joint.capitalize()} angle")
        plt.legend()
        return plt.show()
    
    else:
        _, y = result
        label = f"{specification}"
        plt.plot(_, y, color="green", label=f"{specification}")
        plt.xlabel("Time (s)")
        plt.ylabel(label)
        plt.title(f"{joint.capitalize()} angle")
        plt.legend()
        return plt.show()

In [ ]:
def stat_angle(joint, stat="mean", trial="Curve_1", specification="Rotation"):
    result = get_angle_data(joint, trial, specification)

    func = getattr(np, stat)
    
    if len(result) == 3:
        _, y_l, y_r = result
        return func(y_l), func(y_r)
    else:
        _, y = result
        return func(y)


In [ ]:
angle_l, angle_r = stat_angle("ankle", stat="max", trial="Straight_T1")

print(f"max angle left = {angle_l}°")

#### Ankle Plots

In [ ]:
plot_angle("ankle", trial="Straight_T1")

In [ ]:
plot_angle("ankle", trial="Curve_1")

In [ ]:
plot_angle("ankle", trial="Curve_4")

#### Hip plots

In [ ]:
plot_angle("hip", trial="Curve_1", specification= "Flexion")
plot_angle("hip", trial="Curve_4", specification= "Flexion")
plot_angle("hip", trial="Straight_T1", specification= "Flexion")

angle_c1l, angle_c1r = stat_angle("hip", stat="max", trial= "Curve_1", specification= "Flexion")
angle_c4l, angle_c4r = stat_angle("hip", stat="max", trial= "Curve_4",specification= "Flexion")
angle_s1l, angle_s1r = stat_angle("hip", stat="max", trial= "Straight_T1",specification= "Flexion")

print(f"max angle left hip on a curve = {np.mean([angle_c1l, angle_c4l])}°")
print(f"max angle right hip on a curve = {np.mean([angle_c1r, angle_c4r])}°")
print(f"max angle left hip on the straight = {angle_s1l}°")
print(f"max angle right hip on the straight = {angle_s1r}°")



In [ ]:
plot_angle("hip", trial="Curve_1", specification= "Adduction")
plot_angle("hip", trial="Straight_T1", specification= "Adduction")
plot_angle("hip", trial="Curve_4", specification= "Adduction")

In [ ]:
plot_angle

#### Pelvis plots


In [ ]:
plot_angle("pelvis", trial="Curve_1", specification= "Tilt")
plot_angle("pelvis", trial="Curve_4", specification= "Tilt")
plot_angle("pelvis", trial="Straight_T1", specification= "Tilt")